In [1]:
%pip install ipython-sql --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sqlite3
import pandas as pd

In [3]:
%load_ext sql

In [4]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [5]:
df = pd.read_csv("AAPL_data.csv")

In [6]:
df.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,MA_20,STD_20,Upper_Band,Lower_Band,RSI,MACD,MACD_signal,MACD_diff
0,2016-09-12 00:00:00-04:00,23.474584,24.176649,23.447141,24.112617,181171200,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-09-13 00:00:00-04:00,24.585996,24.878714,24.524250,24.686617,248704800,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-09-14 00:00:00-04:00,24.864989,25.848336,24.835259,25.560192,443554800,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-09-15 00:00:00-04:00,26.038149,26.465792,25.953535,26.429201,359934400,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-09-16 00:00:00-04:00,26.326290,26.557261,26.079309,26.280552,319547600,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Create (or connect to) a database file
conn = sqlite3.connect("stocks.db")

# Save the DataFrame as a table
df.to_sql("stock", conn, if_exists="replace", index=False)

conn.close()

In [8]:
%sql sqlite:///stocks.db

Question 1: What is the highest closing price ever?

Where the market was willing to pay the most — the peak of optimism.

In [9]:
# pandas
df['Close'].max()

339.78692626953125

In [10]:
%%sql
SELECT max(Close)
FROM stock;

 * sqlite:///stocks.db
Done.


max(Close)
339.78692626953125


Question 2: What is the lowesst closing price ever? 

Peak fear / capitulation point. Establishes a floor of despair. If the stock revisits it, market is questioning the company's survival.

In [11]:
# pandas
df['Close'].min()

24.11261749267578

In [12]:
%%sql
Select min(Close)
from stock;

 * sqlite:///stocks.db
Done.


min(Close)
24.11261749267578


Question 3: On which date was the stock at its peak?

 When the crowd was most enthusiastic. Anchors behavioral memory — investors still holding from that date feel pain. This creates resistance.

In [13]:
# pandas
df.loc[df['Close'].idxmax()].Date

'2026-07-28 00:00:00-04:00'

In [14]:
%%sql
select Date
from stock
order by Close desc limit 1;

 * sqlite:///stocks.db
Done.


Date
2026-07-28 00:00:00-04:00


Question 4: What is the average closing price?
The "fair value" over the period

In [15]:
df["Close"].mean()

132.02770385392913

In [16]:
%%sql
select avg(close)
from stock;

 * sqlite:///stocks.db
Done.


avg(close)
132.02770385392913


Question 5: What is the median closing price?

The "typical" price without outliers

In [18]:
df['Close'].median()

136.76878356933594

In [ ]:
%%sql 
select avg(close) as median
from (
    select close 
    from stock 
    order by close 
    limit 2 - (select count(*) from stock) % 2 
    offset (select (count(*) - 1) / 2 from stock));

 * sqlite:///stocks.db
Done.


median
136.76878356933594


Question 6: What is the daily price range (High - Low)

Intraday volatility

In [22]:
(df['High'] - df['Low']).mean()

2.783437928362192

In [23]:
%%sql 
select avg(high - low) from stock

 * sqlite:///stocks.db
Done.


avg(high - low)
2.7834379283621895


Question 7: What is the intraday volatility (std of range)?

How unstable the daily battle is

In [25]:
(df['High'] - df['Low']).std()

2.3982272658597488

In [27]:
%%sql 
select stdev(high - low) from stock;

 * sqlite:///stocks.db
(sqlite3.OperationalError) no such function: stdev
[SQL: select stdev(high - low) from stock;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
